In [1]:
# Install a pip comtradeapicall package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install --upgrade comtradeapicall

In [2]:
# Install a pip pandas package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install pandas

In [3]:
# Install a pip oracledb package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install oracledb

In [4]:
#Import Required Packages
import pandas
import comtradeapicall
import requests
import oracledb

#Subscription Key for ComTradeData
subscription_key = '5ac78a346071428abfbc114525a1fb99'

In [5]:
#Create Empty DataFrame
tradeData = pandas.DataFrame()

#User Defined Function to iterate all the months
def ref_period(year):
    return ','.join(f"{year}{month:02d}" for month in range(1, 13))

# Loop through the range of years from 2017 to 2023
for year in range(2017, 2024):

    #Chosen SouthAfrica, Kenya and Egypt Reporters
    tradeDataImport = comtradeapicall.getFinalData(subscription_key, typeCode='C', freqCode='M', clCode='HS', period = ref_period(year),
                                    reporterCode='404,710,818', cmdCode='7108,710811,710812,710813,710820', flowCode='X', partnerCode=None,
                                    partner2Code=None,
                                    customsCode=None, motCode=None,
                                                includeDesc=True)

    print(f"Year: {year}, Response: {tradeDataImport}")

    tradeData = pandas.concat([tradeData, tradeDataImport], ignore_index=True)

tradeData.reset_index(drop=True, inplace=True)

Year: 2017, Response:     typeCode freqCode  refPeriodId  refYear  refMonth  period  reporterCode  \
0          C        M     20170101     2017         1  201701           404   
1          C        M     20170101     2017         1  201701           404   
2          C        M     20170101     2017         1  201701           404   
3          C        M     20170101     2017         1  201701           404   
4          C        M     20170101     2017         1  201701           404   
..       ...      ...          ...      ...       ...     ...           ...   
919        C        M     20171201     2017        12  201712           818   
920        C        M     20171201     2017        12  201712           818   
921        C        M     20171201     2017        12  201712           818   
922        C        M     20171201     2017        12  201712           818   
923        C        M     20171201     2017        12  201712           818   

    reporterISO reporterDesc 

/var/folders/3v/8q0ps1q12w76v79bg_nnyvmh0000gn/T/ipykernel_4840/779579783.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tradeData = pandas.concat([tradeData, tradeDataImport], ignore_index=True)


In [6]:
#Convert Column Name to UpperCase for Table Insert
tradeData.columns = tradeData.columns.str.upper()

print(tradeData.columns)

Index(['TYPECODE', 'FREQCODE', 'REFPERIODID', 'REFYEAR', 'REFMONTH', 'PERIOD',
       'REPORTERCODE', 'REPORTERISO', 'REPORTERDESC', 'FLOWCODE', 'FLOWDESC',
       'PARTNERCODE', 'PARTNERISO', 'PARTNERDESC', 'PARTNER2CODE',
       'PARTNER2ISO', 'PARTNER2DESC', 'CLASSIFICATIONCODE',
       'CLASSIFICATIONSEARCHCODE', 'ISORIGINALCLASSIFICATION', 'CMDCODE',
       'CMDDESC', 'AGGRLEVEL', 'ISLEAF', 'CUSTOMSCODE', 'CUSTOMSDESC',
       'MOSCODE', 'MOTCODE', 'MOTDESC', 'QTYUNITCODE', 'QTYUNITABBR', 'QTY',
       'ISQTYESTIMATED', 'ALTQTYUNITCODE', 'ALTQTYUNITABBR', 'ALTQTY',
       'ISALTQTYESTIMATED', 'NETWGT', 'ISNETWGTESTIMATED', 'GROSSWGT',
       'ISGROSSWGTESTIMATED', 'CIFVALUE', 'FOBVALUE', 'PRIMARYVALUE',
       'LEGACYESTIMATIONFLAG', 'ISREPORTED', 'ISAGGREGATE'],
      dtype='object')


In [7]:
#DB Connectivity Setup
connection = oracledb.connect(user='"C##goldTrade"', password='goldTrade', dsn='localhost:1521/ORCLCDB')

#Creation of Curser
cursor = connection.cursor()

#Delete Existing data from Data Lake Table
delete_sql = "DELETE FROM gold_trade_dl"
cursor.execute(delete_sql)
connection.commit()

# Generate SQL insert statement dynamically based on columns
tbl_col = tradeData.columns.tolist()
tbl_insert = f"INSERT INTO gold_trade_dl ({', '.join(tbl_col)}) VALUES ({', '.join([':' + str(i+1) for i in range(len(tbl_col))])})"

# Convert DataFrame to list of tuples with correct data types
data = []
for i, row in tradeData.iterrows():
    data.append(tuple([val if pandas.notna(val) else None for val in row]))

#Load Data to Data Lake Table 
cursor.executemany(tbl_insert, data)
connection.commit()

In [8]:
#Source DataFrame detail after initial Load
tradeData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4284 entries, 0 to 4283
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   TYPECODE                  4284 non-null   object 
 1   FREQCODE                  4284 non-null   object 
 2   REFPERIODID               4284 non-null   int64  
 3   REFYEAR                   4284 non-null   int64  
 4   REFMONTH                  4284 non-null   int64  
 5   PERIOD                    4284 non-null   object 
 6   REPORTERCODE              4284 non-null   int64  
 7   REPORTERISO               4284 non-null   object 
 8   REPORTERDESC              4284 non-null   object 
 9   FLOWCODE                  4284 non-null   object 
 10  FLOWDESC                  4284 non-null   object 
 11  PARTNERCODE               4284 non-null   int64  
 12  PARTNERISO                4284 non-null   object 
 13  PARTNERDESC               4284 non-null   object 
 14  PARTNER2

In [9]:
# Function to fetch JSON data from URL, convert to DataFrame, and rename columns
def fetch_uri_data(url, required_columns,new_column_names):
    response = requests.get(url)
    data = response.json()
    mapping_dict = {}
    
    for item in data['results']:
        key = item[required_columns[0]]  # Use the first column as the key
        value = '|'.join([str(item[col]) for col in required_columns[1:]])

        mapping_dict[key] = value
    
    return mapping_dict

# List of URLs with their descriptions, desired column names, and columns to drop
urls = {
    "ReporterCodes": (
        "https://comtradeapi.un.org/files/v1/app/reference/Reporters.json",
        ["reporterCode","reporterDesc","reporterCodeIsoAlpha3"],
        ["REPORTERCODE","REPORTERDESC","REPORTERISO"]
    ),
    "CustomsCodes": (
        "https://comtradeapi.un.org/files/v1/app/reference/CustomsCodes.json",
        ["id","text"],
        ["CUSTOMSCODE", "CUSTOMSDESC"]
    ),
    "ModeOfTransportCodes": (
        "https://comtradeapi.un.org/files/v1/app/reference/ModeOfTransportCodes.json",
        ["id","text"],
        ["MOTCODE", "MOTDESC"]
    ),
    "PartnerCodes": (
        "https://comtradeapi.un.org/files/v1/app/reference/partnerAreas.json",
        ["PartnerCode","PartnerDesc","PartnerCodeIsoAlpha3"],
        ["PARTNERCODE", "PARTNERDESC", "PARTNERISO"]
    ),
    "CommodityCodes": (
        "https://comtradeapi.un.org/files/v1/app/reference/HS.json",
        ["id","text","parent","isLeaf"],
        ["CMDCODE","CMDDESC","PARENT","ISLEAF"]
    )
}

# Fetching data and processing in a dictionary [Dictionary Comprehension]
dict = {
    name: fetch_uri_data(url, cols, drop_cols) 
    for name, (url, cols, drop_cols) in urls.items()
}

#Access DataFrame
ReporterData = dict["ReporterCodes"]
CustomsData = dict["CustomsCodes"]
MotData = dict["ModeOfTransportCodes"]
Partnerdata = dict["PartnerCodes"]
CommodityData = dict["CommodityCodes"]

for key, value in CommodityData.items():
    position_of_hyphen = value.find("-")
    if position_of_hyphen != -1:  # If a hyphen is found
        CommodityData[key] = value[position_of_hyphen + 1:].strip()

In [10]:
#Initialize variables
tradeDataReject = []
rejected_rows = pandas.DataFrame()

#Trim Additional Spaces in All Object columns
tradeData = tradeData.apply(lambda x: x.str.strip() if isinstance(x, str) else x,axis=1)

# Define a function to handle splitting based on the type of value
def split_value(value, index):
    if isinstance(value, list):
        return value[index]
    else:
        return value.split('|')[index]

# Preprocess data to map required values for null values in required columns
tradeData['REPORTERDESC'] = tradeData['REPORTERCODE'].map(lambda code: split_value(Partnerdata.get(code, ''), 0))
tradeData['REPORTERISO'] = tradeData['REPORTERCODE'].map(lambda code: split_value(Partnerdata.get(code, ''), 1))
tradeData['CUSTOMSDESC'] = tradeData['CUSTOMSCODE'].map(lambda code: split_value(CustomsData.get(code, ''), 0))
tradeData['MOTDESC'] = tradeData['MOTCODE'].astype(str).map(lambda code: split_value(MotData.get(code, ''), 0))
tradeData['CMDDESC'] = tradeData['CMDCODE'].astype(str).map(lambda code: split_value(CommodityData.get(code, ''), 0))
tradeData['PARTNERDESC'] = tradeData['PARTNERCODE'].map(lambda code: split_value(Partnerdata.get(code, ''), 0))
tradeData['PARTNERISO'] = tradeData['PARTNERCODE'].map(lambda code: split_value(Partnerdata.get(code, ''), 1))
tradeData['PARTNER2DESC'] = tradeData['PARTNER2CODE'].map(lambda code: split_value(Partnerdata.get(code, ''), 0))
tradeData['PARTNER2ISO'] = tradeData['PARTNER2CODE'].map(lambda code: split_value(Partnerdata.get(code, ''), 1))
tradeData.loc[(tradeData['FLOWCODE'] == 'X'), 'FLOWDESC'] = 'Export'
tradeData.loc[(tradeData['CLASSIFICATIONCODE'].isin(['H0','H1','H2','H3','H4','H5','H6'])), 'CLASSIFICATIONSEARCHCODE'] = 'HS'
tradeData.loc[(tradeData['PRIMARYVALUE'] != tradeData['FOBVALUE'] ) & (tradeData['FOBVALUE'] != 0), 'PRIMARYVALUE'] = tradeData['FOBVALUE']
tradeData.loc[(tradeData['NETWGT'] == 0 ) & (tradeData['QTYUNITCODE'] == 8) & (tradeData['QTY'] != 0 ), 'NETWGT'] = tradeData['QTY']
tradeData.loc[((tradeData['NETWGT'] == 0) | (tradeData['NETWGT'].isnull()) ) & (tradeData['ALTQTYUNITCODE'] == 8) & (tradeData['QTYUNITCODE'] != 8), 'NETWGT'] = tradeData['ALTQTY']
tradeData.loc[(tradeData['NETWGT'] == 0 ) & (tradeData['ALTQTYUNITCODE'] == 15) & (tradeData['QTYUNITCODE'] != 8 ), 'NETWGT'] = tradeData['ALTQTY']/1000
tradeData.loc[(tradeData['ALTQTYUNITCODE'] == 15 ) & (tradeData['QTYUNITCODE'] == 8), 'NETWGT'] = tradeData['ALTQTY']/1000


In [11]:
#Null Condition Check in required columns
for col in ['TYPECODE', 'FREQCODE', 'REFPERIODID', 'PERIOD', 'REPORTERCODE','FLOWCODE', 'PARTNERCODE', 'CLASSIFICATIONCODE', 'ISORIGINALCLASSIFICATION', 'CMDCODE', 'ISLEAF', 'CUSTOMSCODE', 'MOSCODE', 'MOTCODE', 'FOBVALUE' ]:
    
    null_chk = tradeData[col].isna() | (tradeData[col] == '')
    
    #Drop the record if null found in mandate columns
    if null_chk.any():
        rejected_rows = tradeData[null_chk].copy()
        rejected_rows['REJECT_REASON'] = f"Null/Blank found for {col}"
        tradeDataReject.append(rejected_rows)
        tradeData.drop(rejected_rows.index, inplace=True)

print(tradeDataReject)

[]


In [12]:
#Function Declaration to reject records based on condition
def dqChecks(tradeData, col, condition, reject_reason):
    condition_chk = tradeData.apply(lambda row: condition(row[col]), axis=1)
    if condition_chk.any():
        rejected_rows = tradeData[condition_chk].copy()
        rejected_rows['REJECT_REASON'] = reject_reason
        return rejected_rows
    return pandas.DataFrame() #Returning Empty DataFrame if no rejects

In [13]:
#Data Quality Checks Condition Parameters
dq_conditions = [
     ('TYPECODE', lambda x: x != 'C', "Typecode not equal to 'C'"),
     ('FREQCODE', lambda x: x != 'M', "Freqcode not equal to 'M'"),
     ('REPORTERCODE', lambda x: (x != 404) and (x != 710) and (x != 818), "Invalid ReporterCode sent from Source"),
     ('REPORTERISO', lambda x: (x.upper() != 'KEN') and (x.upper() != 'ZAF') and (x.upper() != 'EGY') if isinstance(x, str) else True, "Invalid ReporterISO sent from source"),
     ('REPORTERDESC', lambda x: (x != 'Kenya') and (x != 'South Africa') and (x != 'Egypt') if isinstance(x, str) else True, "Invalid ReporterDesc sent from source"),
    ('FLOWCODE', lambda x: x != 'X', "FlowCode not equal to 'X'"),
    ('FLOWDESC', lambda x: x != 'Export', "Invalid FlowDesc sent from Source"),
    ('PARTNERCODE', lambda x: x == 0, "PartnerCode equals to 0"),
    ('PARTNERISO', lambda x: x == 'W00', "Invalid PartnerISO sent from Source"),
    ('PARTNERDESC', lambda x: x == 'World', "Invalid PartnerDesc sent from Source"),
    ('PARTNER2CODE', lambda x: x != 0, "Partner2Code not equal to 0"),
    ('PARTNER2ISO', lambda x: x != 'W00', "Invalid Partner2ISO sent from Source"),
    ('PARTNER2DESC', lambda x: x != 'World', "Invalid Partner2Desc sent from Source"),
    ('CLASSIFICATIONCODE', lambda x: (x != 'H0') and (x != 'H1') and (x != 'H2') and (x != 'H3') and (x != 'H4') and (x != 'H5') and (x != 'H6'), "Invalid ClassificationCode sent from Source"),
    ('CLASSIFICATIONSEARCHCODE', lambda x: x != 'HS', "Invalid ClassificationSearchCode sent from Source"),
    ('ISORIGINALCLASSIFICATION', lambda x: str(x).strip() != 'True', "Source Data does not belong to Original Classification"),
    ('CMDCODE', lambda x: (x != '7108') and (x != '710811') and (x != '710812') and (x != '710813') and (x != '710820'), "Invalid CmdCode sent from Source"),
    ('CMDDESC', lambda x: (x != 'Gold (including gold plated with platinum) unwrought or in semi-manufactured forms, or in powder form') and (x != 'Metals; gold, non-monetary, powder') and (x != 'Metals; gold, non-monetary, unwrought (but not powder)') and (x != 'Metals; gold, semi-manufactured') and (x != 'Gold, monetary'), "Invalid CmdDesc sent from Source"),
     ('ISLEAF', lambda x: x != 1, "Invalid isLeaf sent from Source"),
     ('ISREPORTED', lambda x: str(x).strip() != 'True', "Invalid Record sent from Source"),
    ('MOSCODE', lambda x: x != '0', "Invalid MosCode sent from Source"),
    ('PRIMARYVALUE', lambda x: (x!=tradeData['FOBVALUE']).all(),"PrimaryValue mismatches with FOBValue"),
    ('NETWGT', lambda x:( x== 0) and (tradeData['QTYUNITCODE'] == -1).all() and (tradeData['ALTQTYUNITCODE'] == -1).all(), "Invalid Netweight sent from source")
]

In [14]:
# Apply condition checks
for col, condition, reject_reason in dq_conditions:
    rejected_rows = dqChecks(tradeData, col, condition, reject_reason)
    if not rejected_rows.empty:
        tradeDataReject.append(rejected_rows)
        tradeData.drop(rejected_rows.index, inplace=True)

print(tradeDataReject)

[     TYPECODE FREQCODE  REFPERIODID  REFYEAR  REFMONTH  PERIOD  REPORTERCODE  \
0           C        M     20170101     2017         1  201701           404   
3           C        M     20170101     2017         1  201701           404   
5           C        M     20170101     2017         1  201701           404   
7           C        M     20170101     2017         1  201701           710   
10          C        M     20170101     2017         1  201701           710   
...       ...      ...          ...      ...       ...     ...           ...   
4270        C        M     20231201     2023        12  202312           710   
4272        C        M     20231201     2023        12  202312           710   
4274        C        M     20231201     2023        12  202312           710   
4276        C        M     20231201     2023        12  202312           818   
4280        C        M     20231201     2023        12  202312           818   

     REPORTERISO  REPORTERDESC FLOWCOD

In [15]:
# Concatenate all rejected DataFrames into a single DataFrame
if tradeDataReject:
    tradeDataRejects = pandas.concat(tradeDataReject).reset_index(drop=True)
else:
    tradeDataRejects = pandas.DataFrame()

In [16]:
#Delete Existing data from Data Reject Table
rejects_sql = "DELETE FROM gold_trade_data_reject"
cursor.execute(rejects_sql)
connection.commit()

# Generate SQL insert statement dynamically based on columns
tbl_col = tradeDataRejects.columns.tolist()
tbl_insert = f"INSERT INTO gold_trade_data_reject ({', '.join(tbl_col)}) VALUES ({', '.join([':' + str(i+1) for i in range(len(tbl_col))])})"

# Convert DataFrame to list of tuples for bulk insert
data = []
for i, row in tradeDataRejects.iterrows():
    data.append(tuple([val if pandas.notna(val) else None for val in row]))

#Load Data to Data Reject Table 
cursor.executemany(tbl_insert, data)
connection.commit()

In [17]:
#Drop unwanted columns
tradeData = tradeData.drop(['TYPECODE','FREQCODE', 'REFPERIODID','REFYEAR','REFMONTH','REPORTERCODE','REPORTERISO','FLOWDESC','PARTNERCODE','PARTNERISO','PARTNER2CODE','PARTNER2ISO','CLASSIFICATIONCODE','CLASSIFICATIONSEARCHCODE','ISORIGINALCLASSIFICATION','CMDDESC','AGGRLEVEL','ISLEAF','CUSTOMSCODE','MOSCODE','MOTCODE','QTYUNITABBR','ISQTYESTIMATED','ALTQTYUNITABBR','ISALTQTYESTIMATED','ISNETWGTESTIMATED','ISGROSSWGTESTIMATED','CIFVALUE','FOBVALUE','LEGACYESTIMATIONFLAG','ISREPORTED','ISAGGREGATE'],axis=1)

#Rename columns based on the given requirements for display
tradeData.rename(columns={'REPORTERDESC':'REPORTER','FLOWCODE':'TRADE_FLOW','PARTNERDESC':'PARTNER','PARTNER2DESC':'2ND_PARTNER','CMDCODE':'COMMODITY_CODE','CUSTOMSDESC':'CUSTOMS_DESC','MOTDESC':'TRANSPORT_MODE','QTYUNITCODE':'QTY_UNIT','ALTQTYUNITCODE':'ALTERNATE_QUANTITY_UNIT','ALTQTY':'ALTERNATE_QUANTITY','NETWGT':'NET_WEIGHT(KG)','GROSSWGT':'GROSS_WEIGHT','PRIMARYVALUE':'TRADE_VALUE(US$)'}, inplace=True)

#Re-order columns in the given order based on requirements
col_order = ['PERIOD','TRADE_FLOW','REPORTER','PARTNER','2ND_PARTNER','CUSTOMS_DESC','TRANSPORT_MODE','COMMODITY_CODE','TRADE_VALUE(US$)','NET_WEIGHT(KG)','GROSS_WEIGHT','QTY_UNIT','QTY','ALTERNATE_QUANTITY_UNIT','ALTERNATE_QUANTITY']

tradeData = tradeData.reindex(columns=col_order)

print(tradeData)

      PERIOD TRADE_FLOW      REPORTER               PARTNER 2ND_PARTNER  \
4     201701          X         Kenya  United Arab Emirates       World   
6     201701          X         Kenya          South Africa       World   
32    201701          X  South Africa               Namibia       World   
44    201701          X  South Africa               Namibia       World   
46    201701          X  South Africa            Areas, nes       World   
...      ...        ...           ...                   ...         ...   
4259  202312          X         Kenya          South Africa       World   
4275  202312          X  South Africa            Areas, nes       World   
4281  202312          X         Egypt                Canada       World   
4282  202312          X         Egypt           Switzerland       World   
4283  202312          X         Egypt  United Arab Emirates       World   

                       CUSTOMS_DESC            TRANSPORT_MODE COMMODITY_CODE  \
4     TOTAL customs

In [18]:
#Delete Existing data from Data Warehouse Table
dw_sql = "DELETE FROM gold_trade"
cursor.execute(dw_sql)
connection.commit()

# Generate SQL insert statement dynamically based on columns
tbl_col = ['"{0}"'.format(col) for col in tradeData.columns.tolist()]
tbl_insert = f"INSERT INTO gold_trade ({', '.join(tbl_col)}) VALUES ({', '.join([':' + str(i+1) for i in range(len(tbl_col))])})"

# Convert DataFrame to list of tuples for bulk insert
data = []
for i, row in tradeData.iterrows():
   data.append(tuple([val if pandas.notna(val) else None for val in row]))

#Load Data to Data Warehouse Table 
cursor.executemany(tbl_insert, data)
connection.commit()

In [19]:
#Drop Unwanted columns for Plotting in Tableau
tradeData = tradeData.drop(['TRADE_FLOW','2ND_PARTNER','GROSS_WEIGHT','QTY_UNIT','QTY','ALTERNATE_QUANTITY_UNIT','ALTERNATE_QUANTITY'],axis=1)

#Delete Existing data from Data Mart Table
dm_sql = "DELETE FROM gold_trade_data_mart"
cursor.execute(dm_sql)
connection.commit()

# Generate SQL insert statement dynamically based on columns
tbl_col = ['"{0}"'.format(col) for col in tradeData.columns.tolist()]
tbl_insert = f"INSERT INTO gold_trade_data_mart ({', '.join(tbl_col)}) VALUES ({', '.join([':' + str(i+1) for i in range(len(tbl_col))])})"

# Convert DataFrame to list of tuples for bulk insert
data = []
for i, row in tradeData.iterrows():
    data.append(tuple([val if pandas.notna(val) else None for val in row]))

#Load Data to Data Mart Table 
cursor.executemany(tbl_insert, data)
connection.commit()

In [20]:
# Clean up: close cursor and connection
cursor.close()
connection.close()